### Singular value decomposition on MNIST example

We reproduce the component-analysis idea from the Yandex ML handbook article on matrix factorization: represent MNIST as a matrix where each row is a flattened image, decompose that matrix with SVD, and visualize the first three latent components.

For a data matrix $X$, SVD gives

$$X = U \Sigma V^T.$$

The three-dimensional coordinates below are the first three columns of $U\Sigma$. Each point is one digit image.

In [15]:
from __future__ import annotations

import numpy as np
import pandas as pd
import plotly.express as px
from numpy.typing import NDArray
from sklearn.datasets import fetch_openml

In [16]:
SAMPLES_PER_CLASS = 100
RANDOM_STATE = 42

### Data

MNIST has 70,000 grayscale images of handwritten digits. Each image is $28 \times 28$, so after flattening every row has 784 pixel features.

In [17]:
def load_mnist() -> tuple[NDArray[np.float64], NDArray[np.int_]]:
    """Load MNIST as a matrix of flattened 28x28 images."""
    mnist = fetch_openml("mnist_784", version=1, as_frame=False)
    images = np.asarray(mnist.data, dtype=np.float64) / 255.0
    labels = np.asarray(mnist.target, dtype=np.int_)
    return images, labels


def sample_per_class(
    images: NDArray[np.float64],
    labels: NDArray[np.int_],
    samples_per_class: int,
    random_state: int,
) -> tuple[NDArray[np.float64], NDArray[np.int_], NDArray[np.int_]]:
    """Take the same number of random examples from every digit class."""
    rng = np.random.default_rng(random_state)
    selected_indices: list[NDArray[np.int_]] = []

    for digit in np.unique(labels):
        digit_indices = np.flatnonzero(labels == digit)
        sample_size = min(samples_per_class, digit_indices.size)
        selected_indices.append(
            rng.choice(digit_indices, size=sample_size, replace=False)
        )

    indices = np.concatenate(selected_indices)
    rng.shuffle(indices)
    return images[indices], labels[indices], indices

In [18]:
images, labels = load_mnist()
images_sample, labels_sample, sample_indices = sample_per_class(
    images,
    labels,
    samples_per_class=SAMPLES_PER_CLASS,
    random_state=RANDOM_STATE,
)

images_sample.shape, labels_sample.shape

((1000, 784), (1000,))

### SVD projection

The first three latent features are computed as `U[:, :3] * singular_values[:3]`. The centered version **subtracts the mean value** of each pixel before decomposition.

In [19]:
def svd_latent_features(
    matrix: NDArray[np.float64],
    n_components: int = 3,
    center: bool = True,
) -> NDArray[np.float64]:
    """Return the first columns of U * Sigma for a data matrix."""
    if n_components < 1:
        msg = "n_components must be positive"
        raise ValueError(msg)

    working_matrix = matrix.copy()
    if center:
        working_matrix -= working_matrix.mean(axis=0, keepdims=True)

    u_matrix, singular_values, _ = np.linalg.svd(
        working_matrix,
        full_matrices=False,
    )
    return u_matrix[:, :n_components] * singular_values[:n_components]


def build_components_frame(
    matrix: NDArray[np.float64],
    labels: NDArray[np.int_],
    sample_indices: NDArray[np.int_],
    center: bool,
) -> pd.DataFrame:
    """Build a tidy table for Plotly visualization."""
    features = svd_latent_features(matrix, n_components=3, center=center)
    preprocessing = "centered pixels" if center else "raw pixels"

    return pd.DataFrame(
        {
            "component_1": features[:, 0],
            "component_2": features[:, 1],
            "component_3": features[:, 2],
            "digit": labels.astype(str),
            "sample_index": sample_indices,
            "preprocessing": preprocessing,
        }
    )

In [20]:
components = pd.concat(
    [
        build_components_frame(
            images_sample,
            labels_sample,
            sample_indices,
            center=False,
        ),
        build_components_frame(
            images_sample,
            labels_sample,
            sample_indices,
            center=True,
        ),
    ],
    ignore_index=True,
)

components.head()

,component_1,component_2,component_3,digit,sample_index,preprocessing
0,4.398899,-3.602733,1.998681,1,40033,raw pixels
1,7.807889,0.693382,1.335587,2,20783,raw pixels
2,4.623971,-0.487644,0.015830,2,46464,raw pixels
3,6.020186,2.490497,1.173014,0,51809,raw pixels
4,4.035733,2.071985,-2.904934,4,21822,raw pixels


### 3D visualization

Each color corresponds to one digit. Use the animation slider to switch between raw pixels and centered pixels, then rotate the 3D chart to inspect the geometry of the first three SVD components.

In [21]:
fig = px.scatter_3d(
    components,
    x="component_1",
    y="component_2",
    z="component_3",
    color="digit",
    animation_frame="preprocessing",
    hover_data=["sample_index"],
    category_orders={"digit": [str(digit) for digit in range(10)]},
    title="MNIST projected onto the first three SVD latent components",
    labels={
        "component_1": "First SVD component",
        "component_2": "Second SVD component",
        "component_3": "Third SVD component",
        "digit": "Digit",
        "preprocessing": "Preprocessing",
    },
    opacity=0.75,
)

fig.update_traces(marker={"size": 4})
fig.update_layout(
    height=750,
    legend_title_text="Digit",
    scene={
        "xaxis_title": "First SVD component",
        "yaxis_title": "Second SVD component",
        "zaxis_title": "Third SVD component",
    },
)
fig.show()

### What Singular Value Decomposition Cannot Do

Singular Value Decomposition can find uncorrelated linear combinations of features that make the largest contribution to variance. For normally distributed data, these directions become the principal axes of the ellipsoid formed by the data cloud. Unfortunately, this superpower of SVD can just as easily turn into a weakness because:

- Data is not always normally distributed. It may have complex geometry, but SVD will stubbornly keep looking for an ellipsoid.

- The most important signal is not always the largest in scale. Forgetting to scale your features is a good way to shoot yourself in the foot when working with SVD.

- The new features are not necessarily easy to interpret. A linear combination of age, work experience, and salary is not exactly something you would want to present to a banking regulator.

- Outliers will almost certainly make your life harder, although SVD may also help you notice them.